# Bi-Temporal Change Detection
**NB 04** | Kyiv, Ukraine 2021→2024 | ChangeFormer

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pathlib, json
import numpy as np
import matplotlib.pyplot as plt
import pygeovision as pgv
from pygeovision.models import get_model
from pygeovision.inference.tiled import TiledInference

client = pgv.PyGeoVision()
print(client)

from pygeovision.models.change_detection.changeformer import ChangeDetection


In [ ]:
BBOX        = (30.48, 50.40, 30.58, 50.50)
DATE_BEFORE = ('2021-04-01', '2021-06-30')
DATE_AFTER  = ('2024-04-01', '2024-06-30')
PROVIDERS   = ['planetary_computer']
BANDS       = ['B02','B03','B04','B08']
DATA_DIR    = pathlib.Path('./results/nb04')
DATA_DIR.mkdir(parents=True, exist_ok=True)

pre_results  = client.search(bbox=BBOX, date_range=DATE_BEFORE, providers=PROVIDERS, cloud_cover_max=10)
post_results = client.search(bbox=BBOX, date_range=DATE_AFTER,  providers=PROVIDERS, cloud_cover_max=10)
print(f"Pre-event : {len(pre_results)} scenes | Post-event: {len(post_results)} scenes")


In [ ]:
PRE_DIR  = DATA_DIR / 'pre';  PRE_DIR.mkdir(exist_ok=True)
POST_DIR = DATA_DIR / 'post'; POST_DIR.mkdir(exist_ok=True)
pre_dl  = client.download(pre_results[:1],  output_dir=str(PRE_DIR),
    bands=BANDS, post_process=['reproject:EPSG:32637','cog'])
post_dl = client.download(post_results[:1], output_dir=str(POST_DIR),
    bands=BANDS, post_process=['reproject:EPSG:32637','cog'])
pre_scene  = pre_dl[0].path  if pre_dl  and pre_dl[0].success  else None
post_scene = post_dl[0].path if post_dl and post_dl[0].success else None
print(f"Pre : {pre_scene}\nPost: {post_scene}")


In [ ]:
# Validate both scenes before preprocessing
for label, path in [('PRE', pre_scene), ('POST', post_scene)]:
    if path:
        rep = client.validator.validate(str(path), required_bands=len(BANDS))
        print(f"{label}: {'PASSED' if rep.passed else 'ISSUES'} | {rep.stats}")


In [ ]:
# Preprocess BOTH with identical parameters — critical for change detection accuracy
PRE_READY  = DATA_DIR / 'pre_preprocessed.tif'
POST_READY = DATA_DIR / 'post_preprocessed.tif'

def preprocess(scene_path, out_path, label):
    scl = next(iter(scene_path.parent.rglob('*SCL*.tif')), None)
    r   = client.prepare_for_ai(str(scene_path), stack_bands=BANDS, bbox=BBOX,
        scl_path=str(scl) if scl else None, normalise='scale_factor',
        scale_factor=10000.0, model_type='change_detection', output_path=str(out_path))
    print(f"{label}: {r['steps']}  shape={r['shape']}  [{r['array'].min():.3f},{r['array'].max():.3f}]")
    return r

if pre_scene and post_scene:
    preprocess(pre_scene.parent / pre_scene.name, PRE_READY,  'PRE ')
    preprocess(post_scene.parent / post_scene.name, POST_READY, 'POST')
    print("Both scenes preprocessed and ready for ChangeFormer")


In [ ]:
CHANGE_MAP = DATA_DIR / 'change_map.tif'
if PRE_READY.exists() and POST_READY.exists():
    cd = ChangeDetection(model_variant='changeformer', in_channels=len(BANDS))
    cd.build()
    res = cd.detect(str(PRE_READY), str(POST_READY), output_path=str(CHANGE_MAP))
    print(f"Change coverage: {res['change_pct']:.2f}%")


In [ ]:
if PREDICTION.exists():
    pred_report = client.validator.validate(str(PREDICTION))
    print("Prediction validation:", "PASSED" if pred_report.passed else f"FIXED — {pred_report.errors}")
    print(pred_report.summary())


In [ ]:
SIEVED  = DATA_DIR / 'change_sieved.tif'
VECTOR  = DATA_DIR / 'change_polygons.geojson'
COG_OUT = DATA_DIR / 'change_cog.tif'
if CHANGE_MAP.exists():
    client.postprocess.sieve_filter(str(CHANGE_MAP), min_pixels=20, output_path=str(SIEVED))
    client.postprocess.vectorise(str(SIEVED), str(VECTOR), target_class=1,
        min_area_m2=100.0, simplify_tolerance=1.0)
    stats = client.postprocess.class_statistics(str(SIEVED))
    client.postprocess.to_cog(str(SIEVED), str(COG_OUT))
    with open(VECTOR) as f: gj = json.load(f)
    print(f"Changed polygons: {len(gj['features'])}")
    print(f"Changed area    : {stats.get(1,{}).get('area_ha',0):.1f} ha")


**NB04 — Change Detection** complete.
- Study area: Kyiv Ukraine
- Sensor: Sentinel-2 4-band
- Model: ChangeFormer MiT-B2
- Pipeline: search → download → validate_raw → preprocess → validate_clean → model → postprocess → COG